In [1]:
# LIBRERIE E PATHS
import os
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.stats import pearsonr

# ----------------------------
# 1) Paths
# ----------------------------
BASE_DIR = r"NCT_analysis\data"  
FILE_BASELINE = os.path.join(BASE_DIR, "states_NCT_from_sEEG.mat")
FILE_WITHIN   = os.path.join(BASE_DIR, "within_phases_states.mat")


In [2]:
# ----------------------------
# 2) Helper: estrazione vettori MATLAB robusta
# ----------------------------
def _to_1d(vec):
    """Converte vari formati MATLAB in array 1D float (850,)."""
    v = np.array(vec).squeeze()
    # Gestione casi in cui MATLAB salva come object/cell con un elemento
    while v.dtype == object:
        v = np.array(v.item()).squeeze()
    return v.astype(float).reshape(-1)

def load_vec(mat_path, var_name):
    d = loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    if var_name not in d:
        # stampa chiavi disponibili per capire il nome corretto
        keys = [k for k in d.keys() if not k.startswith("__")]
        raise KeyError(f"'{var_name}' non trovato in {mat_path}. Variabili disponibili: {keys}")
    return _to_1d(d[var_name])

In [4]:
# ----------------------------
# 3) Carica vettori
# ----------------------------
x0_interictal = load_vec(FILE_BASELINE, "x_interictal_80_250")
xf_ictal      = load_vec(FILE_BASELINE, "x_ictal_80_250")

x01 = load_vec(FILE_WITHIN, "x_preictal1_80_250")
x02 = load_vec(FILE_WITHIN, "x_preictal2_80_250")
xf1 = load_vec(FILE_WITHIN, "x_ictal1_80_250")
xf2 = load_vec(FILE_WITHIN, "x_ictal2_80_250")

# ----------------------------
# 4) Maschera ROI non-zero (77 valori)
# ----------------------------
# Consiglio: usa l'unione dei non-zero dei baseline (più robusto)
mask = (np.abs(x0_interictal) > 0) | (np.abs(xf_ictal) > 0)
idx  = np.flatnonzero(mask)

print(f"Numero di ROI selezionate: {len(idx)}")
# opzionale: se ti aspetti esattamente 77
# assert len(idx) == 77, f"Attese 77 ROI non-zero, trovate {len(idx)}"

def select77(v): 
    return v[idx]

x0b = select77(x0_interictal)
xfb = select77(xf_ictal)

# ----------------------------
# 5) Metriche di confronto
# ----------------------------
def cosine_sim(a, b):
    den = (np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.dot(a, b) / den) if den != 0 else np.nan

def metrics(v, ref):
    v = np.asarray(v); ref = np.asarray(ref)
    d = v - ref
    # Pearson richiede varianza non nulla
    r = pearsonr(v, ref)[0] if (np.std(v) > 0 and np.std(ref) > 0) else np.nan
    return {
        "pearson_r": r,
        "cosine": cosine_sim(v, ref),
        "RMSE": float(np.sqrt(np.mean(d**2))),
        "MAE": float(np.mean(np.abs(d))),
        "mean_delta": float(np.mean(d)),
        "std_delta": float(np.std(d)),
        "max_abs_delta": float(np.max(np.abs(d))),
    }

# ----------------------------
# 6) Confronti: PRE vs baseline interictal, ICT vs baseline ictal
# ----------------------------
pre_vectors = {
    "x01 (pre early)": select77(x01),
    "x02 (pre late)" : select77(x02),
}

ict_vectors = {
    "xf1 (ict early)": select77(xf1),
    "xf2 (ict late)" : select77(xf2),
}

rows = []

# PRE: confronta solo contro baseline interictal
for name, v in pre_vectors.items():
    m = metrics(v, x0b)
    rows.append({"vector": name, "ref": "x0_interictal", **m})

# ICT: confronta solo contro baseline ictal
for name, v in ict_vectors.items():
    m = metrics(v, xfb)
    rows.append({"vector": name, "ref": "xf_ictal", **m})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# ---------------------------------
# 7) Differenze dentro fase (solo 77 ROI)
# ---------------------------------
# differenza dentro fase (solo 77 ROI)
pre_delta = select77(x02) - select77(x01)
ict_delta = select77(xf2) - select77(xf1)

print("||x02-x01|| =", np.linalg.norm(pre_delta))
print("||xf2-xf1|| =", np.linalg.norm(ict_delta))

d_pre = select77(x02) - select77(x01)
d_ict = select77(xf2) - select77(xf1)

print("cos(delta_pre, delta_ict) =", cosine_sim(d_pre, d_ict))
print("pearson(delta_pre, delta_ict) =", pearsonr(d_pre, d_ict)[0])
print("max|delta_pre - delta_ict| =", np.max(np.abs(d_pre - d_ict)))

Numero di ROI selezionate: 50
         vector           ref  pearson_r   cosine     RMSE      MAE  mean_delta  std_delta  max_abs_delta
x01 (pre early) x0_interictal   0.987983 0.987981 0.154966 0.075811    0.004193   0.154909       0.830710
 x02 (pre late) x0_interictal   0.993914 0.993904 0.112157 0.063113    0.004886   0.112051       0.566047
xf1 (ict early)      xf_ictal   0.740214 0.740143 0.694293 0.551029   -0.016386   0.694100       1.831891
 xf2 (ict late)      xf_ictal   0.730035 0.729978 0.715690 0.563650   -0.015692   0.715518       2.082166
||x02-x01|| = 1.3779699263650027
||xf2-xf1|| = 1.3779699263650027
cos(delta_pre, delta_ict) = 0.9999999999999999
pearson(delta_pre, delta_ict) = 1.0
max|delta_pre - delta_ict| = 0.0
